# 03 · Claude API playground
Scratchpad for **Days 15–18**. Needs `ANTHROPIC_API_KEY`. Every cell that calls the API costs real tokens: set `LAB_MODEL=claude-haiku-5-5` for cheap practice.

In [1]:
import os, anthropic
MODEL = os.environ.get('LAB_MODEL', 'claude-opus-5-5')
HAS_KEY = bool(os.environ.get('ANTHROPIC_API_KEY'))
client = anthropic.Anthropic() if HAS_KEY else None
print('model:', MODEL, '| API key set:', HAS_KEY)

model: claude-opus-5-5 | API key set: False


## 1. One call

In [2]:
if HAS_KEY:
    r = client.messages.create(model=MODEL, max_tokens=2000, output_config={'effort': 'low'},
        messages=[{'role': 'user', 'content': 'Explain SCD type 2 in 3 bullet points for a data engineer.'}])
    print(r.stop_reason, r.usage.input_tokens, r.usage.output_tokens)
    print(''.join(b.text for b in r.content if b.type == 'text'))

## 2. Structured output: SQL → lineage JSON

In [3]:
from pydantic import BaseModel
class Lineage(BaseModel):
    source_tables: list[str]
    output_columns: list[str]
    filters: list[str]

SQL = open('../../dbt_project/models/marts/fct_orders.sql').read()
if HAS_KEY:
    r = client.messages.parse(model=MODEL, max_tokens=4000, output_format=Lineage,
        messages=[{'role': 'user', 'content': 'Extract lineage from this dbt model:\n' + SQL}])
    print(r.parsed_output)

## 3. Prompt caching check
Run the same long-prefix request twice; the second should show `cache_read_input_tokens > 0`.

In [4]:
GUIDE = open('../reference/ticket_classification_guide.md').read() * 3   # long, stable prefix
if HAS_KEY:
    for i in range(2):
        r = client.messages.create(model=MODEL, max_tokens=500, output_config={'effort': 'low'},
            system=[{'type': 'text', 'text': GUIDE, 'cache_control': {'type': 'ephemeral'}}],
            messages=[{'role': 'user', 'content': 'order 271002 was 50 min late'}])
        print(i, 'written:', r.usage.cache_creation_input_tokens, 'read:', r.usage.cache_read_input_tokens)